# Suite CPRT — Printing C translation units from the store

`Programs.C.Print` prints a translation unit as the store holds it: a reference object that
`Programs.Framework.TheWorkspace` owns by its path, holding its tree as one value, its `body`. Each declaration and
statement is a value object, a branch of the union `Programs.C.Statement` tagged by its kind in lower case, holding
its blocks and parts as its properties, in their containers' order; its expressions are mbse-expressions' C family's
terms and helpers, which `C.Expressions.Text.Printer` writes and `Print.Printer` extends: where a statement expression,
a struct or a union, or a conditional directive holds statements, members or a group (its `block`, `members` or
`group`), they print within the term, at the level of the statement that holds it, to any depth. The layout is the 0.3
printer's: four spaces a level, braces on the line that opens them, `} else {` and `else if`, a statement that is no
block on the next line one level in, `case` labels one level in from their `switch` and their statements one further,
other labels one level out, directives at the start of their lines, comments as written, a trailing one on the line
it ends.

Each case builds its translation unit in a transaction of its own, its tree written in the builders' own syntax: the
unit's `body` a list of the union's branches, a union's branch by its tag (`lambda u: u.ifstatement(...)`), the terms
and helpers nested in it as values of mbse-expressions' unions (`a` below writes the variable `a`, `int` the specifier
`int`). It checks the printed text. The cases port 0.3's C cases of PRT (`04_Print.ipynb`), its expected texts kept
where the layout is unchanged; its cases of expressions alone (precedence, declarators' parentheses) are
mbse-expressions' own suite's, the expressions' printer being its. The case setup is below.

In [ ]:
from mbse.Expressions.Framework import Objects, Terms
from mbse.Programs import C
from mbse.Programs.C import Print, Statements
from mbse.Programs.Framework import Model
from support import raises

store = C.store()
R = store.registered


def build(path, body):
    """The translation unit at `path`, built in a transaction of its own: `body` its declarations' writers."""
    store.transaction(lambda t: t.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"ws": R("Programs.Framework.Workspace"), "unit": R("Programs.C.TranslationUnit")})
        .spec([lambda _: _.ws.files({path: lambda e: e.file(_.unit)}), lambda _: _.unit.body(body)])]))]))


def printed(path, body):
    """The text of the translation unit built at `path`."""
    build(path, body)
    return Print.print_module(store, path)


def within(path, statements):
    """The text of the function `void f(void)` whose body holds `statements`, built at `path`."""
    return printed(path, [function("f", statements)])


# The writers the cases share, in the builders' syntax: names, terms, specifiers, declarators and statements
def var(name):
    return lambda t: t.variable(lambda v: v.name(name))


def number(value):
    return lambda t: t.literal(lambda l: l.value(value))


def name(spelling):
    return lambda i: i.spelling(spelling)


def keyword(word):
    return lambda s: s.primitivetypespecifier(lambda p: p.keyword(word))


def declared(spelling):
    return lambda d: d.iddeclarator(lambda i: i.name(lambda n: n.identifier(name(spelling))))


def call(function_, *arguments):
    return lambda t: t.callexpression(lambda c: c.function(var(function_)).arguments(list(arguments)))


def say(function_):
    """The statement `function_();`."""
    return lambda u: u.expressionstatement(lambda s: s.expression(call(function_)))


def function(spelling, statements, returns="void"):
    """`returns spelling(void) { statements }`."""
    void = lambda p: p.parameterdeclaration(lambda q: q.specifiers([keyword("void")]))  # noqa: E731
    return lambda u: u.functiondefinition(lambda f: f.specifiers([keyword(returns)]).declarator(
        lambda d: d.functiondeclarator(lambda g: g.declarator(declared(spelling)).parameters([void])))
        .body(lambda k: k.body(statements)))


a, b, c = var("a"), var("b"), var("c")
integer = keyword("int")
compound = lambda statements: lambda k: k.compoundstatement(lambda s: s.body(statements))  # noqa: E731

<a id="CPRT-01"></a>
## CPRT-01 · Every simple statement and declaration kind prints alone, on its line

A comment, `//` and `/* */`; an expression statement, and `;`; `break`, `continue`, `return` with a value and
without, `goto` a label and GNU's computed `goto`; declarations with their specifiers, attributes and declarators,
each with what initializes it; `static_assert` with a message and without; an attribute declaration; and `asm`, with
its qualifiers and its operands, outputs, inputs, clobbers and labels, and without; `_Pragma`, a directive where a
statement or a declaration stands, at its level, and a term within an expression; GNU's labels as values, `&&done`,
and asm labels, `int x asm("r1") = 3`, after the whole declarator.

In [ ]:
text = within("simple.c", [
    lambda u: u.comment(lambda s: s.text(" a note")),
    lambda u: u.comment(lambda s: s.block(True).text(" a block ")),
    lambda u: u.expressionstatement(lambda s: s.expression(call("f", a, b))),
    lambda u: u.expressionstatement(lambda s: s),
    lambda u: u.breakstatement(lambda s: s),
    lambda u: u.continuestatement(lambda s: s),
    lambda u: u.returnstatement(lambda s: s),
    lambda u: u.returnstatement(lambda s: s.value(lambda t: t.binaryexpression(lambda x: x.left(a).operator(",")
                                                                               .right(b)))),
    lambda u: u.gotostatement(lambda s: s.label(name("done"))),
    lambda u: u.gotostatement(lambda s: s.target(lambda t: t.subscriptexpression(lambda x: x.object(a).indices([b])))),
    lambda u: u.simpledeclaration(lambda s: s.attributes([lambda a_: a_.standardattributespecifier(
        lambda x: x.attributes([lambda y: y.name("maybe_unused")]))]).specifiers([
        lambda s_: s_.declspecifier(lambda d: d.keyword("static")), lambda s_: s_.cvqualifier(lambda q: q.keyword(
            "const")), integer]).declarators([
        lambda d: d.declarator(declared("x")).initializer(number(1)),
        lambda d: d.declarator(lambda p: p.pointerdeclarator(lambda q: q.declarator(declared("p")))),
        lambda d: d.declarator(lambda p: p.arraydeclarator(lambda q: q.declarator(declared("v")).size(number(2))))
        .initializer(lambda t: t.initializerlist(lambda l: l.items([number(1), number(2)])))])),
    lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda s_: s_.declspecifier(lambda d: d.keyword("typedef")),
                                                          integer]).declarators([lambda d: d.declarator(declared("T"))])),
    lambda u: u.staticassertdeclaration(lambda s: s.keyword("static_assert").condition(a)),
    lambda u: u.staticassertdeclaration(lambda s: s.keyword("_Static_assert").condition(a).message(
        lambda t: t.literal(lambda l: l.value("why")))),
    lambda u: u.attributedeclaration(lambda s: s.attributes([lambda a_: a_.standardattributespecifier(
        lambda x: x.attributes([lambda y: y.name("fallthrough")]))])),
    lambda u: u.asmdeclaration(lambda s: s.keyword("asm").template(lambda t: t.literal(lambda l: l.value("nop")))),
    lambda u: u.asmdeclaration(lambda s: s.keyword("__asm__").volatile(True).goto(True).template(
        lambda t: t.literal(lambda l: l.value("mov %1, %0"))).outputs([
        lambda o: o.name(name("out")).constraint(lambda t: t.literal(lambda l: l.value("=r"))).value(a)]).inputs([
        lambda o: o.constraint(lambda t: t.literal(lambda l: l.value("r"))).value(b)]).clobbers([
        lambda t: t.literal(lambda l: l.value("memory"))]).labels([name("done")])),
    lambda u: u.asmdeclaration(lambda s: s.keyword("asm").inline(True).template(lambda t: t.literal(
        lambda l: l.value("x"))).inputs([lambda o: o.constraint(lambda t: t.literal(lambda l: l.value("r"))).value(b)])),
])
assert text == """\
void f(void) {
    // a note
    /* a block */
    f(a, b);
    ;
    break;
    continue;
    return;
    return a, b;
    goto done;
    goto *a[b];
    [[maybe_unused]] static const int x = 1, *p, v[2] = {1, 2};
    typedef int T;
    static_assert(a);
    _Static_assert(a, "why");
    [[fallthrough]];
    asm("nop");
    __asm__ volatile goto("mov %1, %0" : [out] "=r" (a) : "r" (b) : "memory" : done);
    asm inline("x" : : "r" (b));
}
"""
# The pragma operator, a directive where an item stands, at its statement's level, and a term within an expression;
# GNU's labels as values; GNU's asm labels, after the whole declarator, its keyword as written
pragma = lambda text: lambda d: d.pragmaexpression(lambda p: p.operand(  # noqa: E731
    lambda t: t.c(lambda c: c.literal(lambda l: l.value(text)))))
label = lambda keyword, text: lambda k: k.keyword(keyword).label(  # noqa: E731
    lambda t: t.literal(lambda l: l.value(text)))
assert printed("gnu.c", [
    lambda u: u.directive(lambda s: s.directive(pragma("once"))),
    lambda u: u.simpledeclaration(lambda s: s.specifiers([integer]).declarators([
        lambda d: d.declarator(lambda k: k.iddeclarator(lambda i: i.name(lambda n: n.identifier(name("x"))).asm(
            label("asm", "r1")))).initializer(number(3)),
        lambda d: d.declarator(lambda p: p.pointerdeclarator(lambda q: q.declarator(lambda k: k.iddeclarator(
            lambda i: i.name(lambda n: n.identifier(name("p"))).asm(label("__asm__", "p2"))))))])),
    function("g", [
        lambda u: u.directive(lambda s: s.directive(pragma("GCC unroll 2"))),
        lambda u: u.expressionstatement(lambda s: s.expression(pragma("x"))),
        lambda u: u.returnstatement(lambda s: s.value(lambda t: t.labeladdressexpression(
            lambda x: x.label(name("done"))))),
        lambda u: u.labeledstatement(lambda s: s.label(name("done"))),
    ], returns="void"),
]) == """\
_Pragma("once")
int x asm("r1") = 3, *p __asm__("p2");
void g(void) {
    _Pragma("GCC unroll 2")
    _Pragma("x");
    return &&done;
done:
}
"""
print("ok")

<a id="CPRT-02"></a>
## CPRT-02 · Every compound statement kind prints, with its substatements and labels

A block, empty and not; `if` and `else`, and `else if`; `while`, `do ... while` with a block and without; `for` with a
declaration, an expression or nothing as its initializer, and its condition and increment; `switch` with `case` and
`default` labels, GNU's case range, a label labeling a label and a block, a label ending its block (C23), and a
`switch` whose body is a label; a labeled statement; and an attributed statement.

In [ ]:
text = within("compound.c", [
    compound([say("b")]),
    compound([]),
    lambda u: u.ifstatement(lambda s: s.condition(a).consequence(compound([say("b")])).alternative(compound([say("c")]))),
    lambda u: u.ifstatement(lambda s: s.condition(a).consequence(say("b")).alternative(
        lambda e: e.ifstatement(lambda s: s.condition(b).consequence(say("c")).alternative(say("d"))))),
    lambda u: u.ifstatement(lambda s: s.condition(a).consequence(say("b"))),
    lambda u: u.whilestatement(lambda s: s.condition(a).body(say("b"))),
    lambda u: u.dostatement(lambda s: s.body(compound([say("b")])).condition(a)),
    lambda u: u.dostatement(lambda s: s.body(say("b")).condition(a)),
    lambda u: u.forstatement(lambda s: s.initializer(lambda i: i.simpledeclaration(lambda d: d.specifiers([integer])
        .declarators([lambda d: d.declarator(declared("i")).initializer(number(0))])))
        .condition(lambda t: t.binaryexpression(lambda x: x.left(var("i")).operator("<").right(a)))
        .increment(lambda t: t.unaryexpression(lambda x: x.operator("++").operand(var("i")))).body(compound([say("b")]))),
    lambda u: u.forstatement(lambda s: s.initializer(lambda i: i.expression(lambda t: t.assignmentexpression(
        lambda x: x.left(a).operator("=").right(b)))).body(lambda k: k.expressionstatement(lambda e: e))),
    lambda u: u.forstatement(lambda s: s.body(compound([]))),
    lambda u: u.switchstatement(lambda s: s.condition(a).body(compound([
        lambda u: u.casestatement(lambda c_: c_.value(number(1)).statement(lambda k: k.casestatement(
            lambda c_: c_.value(number(2)).statement(say("b"))))),
        lambda u: u.breakstatement(lambda s: s),
        lambda u: u.casestatement(lambda c_: c_.value(number(3)).last(number(5)).statement(compound([
            lambda u: u.breakstatement(lambda s: s)]))),
        lambda u: u.defaultstatement(lambda d: d.statement(say("c"))),
        lambda u: u.casestatement(lambda c_: c_.value(number(6))),
    ]))),
    lambda u: u.switchstatement(lambda s: s.condition(a).body(lambda k: k.casestatement(
        lambda c_: c_.value(number(1)).statement(say("b"))))),
    lambda u: u.labeledstatement(lambda s: s.label(name("again")).statement(say("b"))),
    lambda u: u.labeledstatement(lambda s: s.label(name("block")).statement(compound([say("c")]))),
    lambda u: u.attributedstatement(lambda s: s.attributes([lambda a_: a_.standardattributespecifier(
        lambda x: x.attributes([lambda y: y.namespace("gnu").name("hot")]))]).statement(compound([]))),
    lambda u: u.attributedstatement(lambda s: s.attributes([lambda a_: a_.gnuattributespecifier(
        lambda x: x.keyword("__attribute__").attributes([lambda y: y.name("fallthrough")]))]).statement(
        lambda k: k.expressionstatement(lambda e: e))),
    lambda u: u.labeledstatement(lambda s: s.label(name("end"))),
])
assert text == """\
void f(void) {
    {
        b();
    }
    {}
    if (a) {
        b();
    } else {
        c();
    }
    if (a)
        b();
    else if (b)
        c();
    else
        d();
    if (a)
        b();
    while (a)
        b();
    do {
        b();
    } while (a);
    do
        b();
    while (a);
    for (int i = 0; i < a; ++i) {
        b();
    }
    for (a = b;;)
        ;
    for (;;) {}
    switch (a) {
        case 1:
        case 2:
            b();
            break;
        case 3 ... 5: {
            break;
        }
        default:
            c();
        case 6:
    }
    switch (a)
        case 1:
            b();
again:
    b();
block: {
    c();
}
    [[gnu::hot]] {}
    __attribute__((fallthrough));
end:
}
"""
print("ok")

<a id="CPRT-03"></a>
## CPRT-03 · Functions, K&R definitions, structs, unions and enumerations: a struct's members within its specifier, one level in, bit-fields and anonymous members, nested to any depth; an enumeration's enumerators

In [ ]:
member = lambda spelling, *rest: lambda m: m.memberdeclaration(lambda d: d.specifiers([integer]).declarators([  # noqa: E731
    lambda x: x.declarator(declared(spelling)), *rest]))
union = lambda s: s.classspecifier(lambda k: k.key("union").body(True).members(lambda m: m.memberlist(  # noqa: E731
    lambda l: l.members([member("i")]))))
struct = lambda s: s.classspecifier(lambda k: k.key("struct").name(lambda n: n.identifier(name("s"))).body(True)  # noqa: E731
    .members(lambda m: m.memberlist(lambda l: l.members([
        member("x", lambda x: x.declarator(declared("y")).bitfield(number(3))),
        lambda m: m.memberdeclaration(lambda d: d.declarators([lambda x: x.bitfield(number(0))]).specifiers([integer])),
        lambda m: m.memberdeclaration(lambda d: d.specifiers([union])),
        lambda m: m.staticassertdeclaration(lambda s: s.keyword("static_assert").condition(number(1))),
        lambda m: m.comment(lambda s: s.text(" last").trailing(True)),
    ]))))
text = printed("types.c", [
    lambda u: u.simpledeclaration(lambda s: s.specifiers([struct]).declarators([lambda d: d.declarator(declared("v"))])),
    lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda s: s.classspecifier(lambda k: k.key("struct")
        .name(lambda n: n.identifier(name("empty"))).body(True))])),
    lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda s: s.enumspecifier(lambda k: k.key("enum")
        .name(lambda n: n.identifier(name("e"))).body(True).enumerators([
            lambda t: t.enumerator(lambda e: e.name(name("A")).value(number(1))),
            lambda t: t.enumerator(lambda e: e.name(name("B")))]).trailing_comma(True))])),
    lambda u: u.functiondefinition(lambda f: f.specifiers([integer]).declarator(lambda d: d.functiondeclarator(
        lambda g: g.declarator(declared("old")).parameters([lambda p: p.parameterdeclaration(lambda q: q.declarator(
            declared("x")))]))).declarations([lambda d: d.specifiers([integer]).declarators([
                lambda d: d.declarator(declared("x"))])]).body(lambda k: k.body([
                    lambda u: u.returnstatement(lambda s: s.value(var("x")))]))),
    function("g", [lambda u: u.simpledeclaration(lambda s: s.specifiers([struct]))], "int"),
])
assert text == """\
struct s {
    int x, y : 3;
    int : 0;
    union {
        int i;
    };
    static_assert(1); // last
} v;
struct empty {};
enum e {
    A = 1,
    B,
};
int old(x)
    int x;
{
    return x;
}
int g(void) {
    struct s {
        int x, y : 3;
        int : 0;
        union {
            int i;
        };
        static_assert(1); // last
    };
}
"""
print("ok")

<a id="CPRT-04"></a>
## CPRT-04 · Directives, at the start of their lines: a conditional's group, its statements or members at their level, between its lines; its `#else`'s and `#elif`'s; an enumeration's directives among its enumerators; comments, trailing and not

In [ ]:
ifdef = lambda group: lambda d: d.ifdefdirective(lambda i: i.name(name("A")).group(group))  # noqa: E731
text = printed("directives.c", [
    lambda u: u.directive(lambda d: d.directive(lambda t: t.includedirective(lambda i: i.path("stdio.h").system(True)))),
    lambda u: u.directive(lambda d: d.directive(lambda t: t.definedirective(lambda i: i.name(name("X")).replacement("1")))),
    lambda u: u.comment(lambda s: s.text(" after").trailing(True)),
    lambda u: u.directive(lambda d: d.directive(lambda t: t.ifdirective(lambda i: i.condition(  # a condition: any
        lambda t: t.c(lambda t: t.definedexpression(lambda e: e.name(name("A"))))).group(  # dialect's term, C's here
            lambda g: g.group(lambda g: g.body([
            lambda u: u.simpledeclaration(lambda s: s.specifiers([integer]).declarators([lambda d: d.declarator(declared("a"))]))])))
        .alternative(lambda t: t.elsedirective(lambda e: e.group(lambda g: g.group(lambda g: g.body([
            lambda u: u.comment(lambda s: s.text(" none"))])))))))),
    function("f", [lambda u: u.directive(lambda d: d.directive(ifdef(lambda g: g.group(lambda g: g.body([say("b")])))))]),
    lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda s: s.classspecifier(lambda k: k.key("struct")
        .name(lambda n: n.identifier(name("s"))).body(True).members(lambda m: m.memberlist(lambda l: l.members([
            lambda m: m.directive(lambda d: d.directive(ifdef(lambda g: g.membergroup(lambda g: g.members([
                lambda m: m.memberdeclaration(lambda d: d.specifiers([integer]).declarators([
                    lambda x: x.declarator(declared("a"))]))]))))),
            lambda m: m.directive(lambda d: d.directive(ifdef(lambda g: g.membergroup(lambda g: g)))),
        ]))))])),
    lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda s: s.enumspecifier(lambda k: k.key("enum")
        .name(lambda n: n.identifier(name("e"))).body(True).enumerators([
            lambda t: t.enumerator(lambda e: e.name(name("A"))),
            lambda t: t.ifdefdirective(lambda i: i.name(name("B")).items([
                lambda t: t.c_types(lambda t: t.enumerator(lambda e: e.name(name("B"))))])),
            lambda t: t.comment(lambda c_: c_.text(" c")),
            lambda t: t.enumerator(lambda e: e.name(name("C")))]))])),
    lambda u: u.directive(lambda d: d.directive(lambda t: t.ifdefdirective(lambda i: i.name(name("Z"))))),
])
assert text == """\
#include <stdio.h>
#define X 1 // after
#if defined(A)
int a;
#else
// none
#endif
void f(void) {
#ifdef A
    b();
#endif
}
struct s {
#ifdef A
    int a;
#endif
#ifdef A
#endif
};
enum e {
    A,
#ifdef B
    B,
#endif
    // c
    C
};
#ifdef Z
#endif
"""
print("ok")

<a id="CPRT-05"></a>
## CPRT-05 · Statements within terms: a statement expression's block, at the level of the statement that holds it, one level in, nested; a struct defined within a cast or a parameter, its members at the level of the statement that holds it

In [ ]:
inner = lambda t: t.statementexpression(lambda s: s.block(lambda k: k.compoundstatement(lambda k: k.body([  # noqa: E731
    lambda u: u.expressionstatement(lambda s: s.expression(a))]))))
outer = lambda t: t.statementexpression(lambda s: s.block(lambda k: k.compoundstatement(lambda k: k.body([  # noqa: E731
    lambda u: u.ifstatement(lambda s: s.condition(b).consequence(compound([
        lambda u: u.expressionstatement(lambda s: s.expression(inner))]))),
    lambda u: u.expressionstatement(lambda s: s.expression(c))]))))
anonymous = lambda s: s.classspecifier(lambda k: k.key("struct").body(True).members(lambda m: m.memberlist(  # noqa: E731
    lambda l: l.members([lambda m: m.memberdeclaration(lambda d: d.specifiers([integer]).declarators([
        lambda x: x.declarator(declared("m"))]))]))))
text = within("terms.c", [
    lambda u: u.returnstatement(lambda s: s.value(outer)),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda t: t.sizeofexpression(lambda z: z.operand(
        lambda t: t.typeid(lambda i: i.specifiers([anonymous])))))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda t: t.statementexpression(lambda s: s))),
])
assert text == """\
void f(void) {
    return ({
        if (b) {
            ({
                a;
            });
        }
        c;
    });
    sizeof(struct {
        int m;
    });
    ({});
}
"""
print("ok")

<a id="CPRT-06"></a>
## CPRT-06 · Invalid syntax prints as written, every kind prints alone, and what is no statement is refused

An `Error` where a statement or a member stands, its text as written, its first line at its level and the others as
the source indented them, and nothing for the statements recovered within it; one with no text, nothing; one where a
`for`'s initializer stands, its text in its place. `print_statement` prints one statement, declaration or member
alone; every kind of `Statements.KINDS`, and the `Error`, has a printer (`Print.KINDS`); a kind none prints is
refused, as is a file that is no translation unit, and a path the workspace holds no file at.

In [ ]:
text = within("invalid.c", [
    lambda u: u.error(lambda e: e.text("if (a {\n  b;\n}").body([say("b")])),
    lambda u: u.error(lambda e: e),
    lambda u: u.forstatement(lambda s: s.initializer(lambda i: i.error(lambda e: e.text("int = ;"))).body(compound([]))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda t: t.error(lambda e: e.text("x = = 1")))),
])
assert text == "void f(void) {\n    if (a {\n  b;\n}\n    for (int = ;;) {}\n    x = = 1;\n}\n"
build("alone.c", [function("f", [say("b")]), lambda u: u.simpledeclaration(lambda s: s.specifiers([struct]).declarators(
    [lambda d: d.declarator(declared("v"))]))])
with store.transaction():
    unit = store.singleton(Model.WORKSPACE).files({"path": "alone.c"})
    definition, declaration = [Terms.kind_of(s)[1] for s in Objects.properties(unit)["body"].values]
    assert Print.print_statement(definition) == "void f(void) {\n    b();\n}"
    specifier = Terms.kind_of(Objects.properties(declaration)["specifiers"].values[0])[1]
    members = Terms.kind_of(Objects.properties(specifier)["members"])[1]
    first = Terms.kind_of(Objects.properties(members)["members"].values[0])[1]
    assert Print.print_statement(first) == "int x, y : 3;"  # a member declaration alone
    with raises(TypeError, match="no printer for the statement kind 'Programs.C.InitDeclarator'"):
        Print.print_statement(Objects.properties(declaration)["declarators"].values[0])
    with raises(TypeError, match="not a Programs.C.TranslationUnit"):
        Print.print_file(store.singleton(Model.WORKSPACE))
assert set(Print.KINDS) == {*Statements.KINDS, "Error"}
with raises(LookupError):
    Print.print_module(store, "nowhere.c")
assert printed("empty.c", []) == ""
print("ok")